In [4]:
import torch

from transformers import BertTokenizer, BertModel, BertForSequenceClassification


In [7]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "Bert is a powerful language model"

inputs = tokenizer(
    text ,
    return_tensors="pt",
    padding= True,
    truncation=True
)

print("Token IDs:")
print(inputs["input_ids"])

print("\n tokens:")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

Token IDs:
tensor([[  101, 14324,  2003,  1037,  3928,  2653,  2944,   102]])

 tokens:
['[CLS]', 'bert', 'is', 'a', 'powerful', 'language', 'model', '[SEP]']


In [9]:
model = BertModel.from_pretrained("bert-base-uncased")
outputs = model(**inputs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
last_hidden_state = outputs.last_hidden_state

print("\nshape of the last_hidden_state:")
print(last_hidden_state.shape)



shape of the last_hidden_state:
torch.Size([1, 8, 768])


In [14]:
cls_vector = last_hidden_state[: , 0 , :]
print("CLS vector shape:")
print(cls_vector.shape)

print("Frist 10 values of the cls vector: ")
print(cls_vector[0][:10])


CLS vector shape:
torch.Size([1, 768])
Frist 10 values of the cls vector: 
tensor([-0.2544, -0.0836,  0.1981, -0.1443, -0.3613, -0.4333,  0.1398,  0.2950,
         0.0778, -0.1298], grad_fn=<SliceBackward0>)


In [20]:
model_with_attention = BertModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

outputs_attention = model_with_attention(**inputs)

attentions = outputs_attention.attentions

print("\nNumber of encoder layers:")
print(len(attentions))

print("\nShape of attention matrix of layer 1:")
print(attentions[0].shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Number of encoder layers:
12

Shape of attention matrix of layer 1:
torch.Size([1, 12, 8, 8])


In [22]:
classifier_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

classification_output = classifier_model(**inputs)

logits = classification_output.logits

print("\nLogits from classifier:")
print(logits)

# Convert logits to prediction
prediction = torch.argmax(logits, dim=1)

print("\nPredicted class:")
print(prediction.item())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Logits from classifier:
tensor([[-0.4829,  0.4501]], grad_fn=<AddmmBackward0>)

Predicted class:
1


In [23]:
probabilities = torch.softmax(logits, dim=1)

print("\nClass probabilities:")
print(probabilities)


Class probabilities:
tensor([[0.2823, 0.7177]], grad_fn=<SoftmaxBackward0>)
